# XGBoost (recency-weighted) vs log-HAR — 6-year window, δ=0.999, MCS test

Does a **gradient-boosted tree** model beat (or join) the best **log-HAR** specs once both are held to
the *same* forecasting protocol? Here every model uses a **6-year rolling window (1512 trading days)**
and **geometric recency decay δ=0.999**, and is scored by **QLIKE in levels** on the **same common
out-of-sample dates** (so only the model class differs).

**Recency in XGBoost.** Trees have no parameter vector, so there is no "inbuilt Kalman filter". The
tree-ensemble analog of weighted least squares is `sample_weight`: each training row contributes
`δ**age` to the loss (newest row weight 1, oldest `δ**(W-1)`), the *identical* geometric decay used by
the HAR's WLS fit (`_recency_weights`). Combined with daily walk-forward refits on the trailing
window, that is how the model "forgets" stale observations. (A Kalman / time-varying-parameter HAR
would be an *alternative to* HAR, not a component of XGBoost — out of scope here.)

**Models compared (all 6y / δ=0.999):**
- `xgb_matched` — XGBoost on the HAR feature set `[x_d, x_w, x_m, log_GVZ, macro]` (head-to-head with Run 18)
- `xgb_rich` — XGBoost also given `log_RV_crude` and `log_RV_ES`
- `har_run18` — log HAR + log GVZ + macro
- `har_run19` — log HAR + log SPX (RV_ES) + log GVZ + macro
- `har_run20` — log HAR + log crude (RV_crude) + log GVZ + macro

The HAR machinery (recency weights, QLIKE, Duan-smearing log→level back-transform, common-OOS gate) is
copied verbatim from `HAR_simpleOLS_3d_with_macro.ipynb`; XGBoost reuses the *same* weights and Duan
smearing so the only difference is OLS → trees.

In [ ]:
# ===========================================================================
# Cell 1 — Imports & data
# ===========================================================================
# NOTE on macOS: the pip xgboost wheel needs an OpenMP runtime (libomp.dylib). If
# `import xgboost` raises an "@rpath/libomp.dylib could not be loaded" error, point it
# at the libomp that ships with scikit-learn, e.g. once per machine:
#   install_name_tool -add_rpath \
#     "$(python3 -c 'import sklearn,os;print(os.path.dirname(sklearn.__file__)+"/.dylibs")')" \
#     "$(python3 -c 'import xgboost,os;print(os.path.dirname(xgboost.__file__)+"/lib/libxgboost.dylib")')"
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from arch.bootstrap import MCS

data = pd.read_parquet("merged_RV_GVZ_with_macro_event.parquet")
rv = data["RV_gold"].astype(float)

TRADING_DAYS = 252
WINDOW_YEARS = np.arange(1.0, 7.001, 0.25)                 # reference grid (defines common OOS)
WINDOWS = [int(round(yr * TRADING_DAYS)) for yr in WINDOW_YEARS]
WINDOW = 6 * TRADING_DAYS                                  # 1512 days = 6y, the only fitting window here
DELTA = 0.999                                             # geometric recency decay
EPS = 1e-6                                                 # QLIKE forecast floor

print(f"RV_gold: {len(rv)} obs, {rv.index.min().date()} .. {rv.index.max().date()}")
print(f"WINDOW={WINDOW} days (6y), DELTA={DELTA}, xgboost {xgb.__version__}")

In [ ]:
# ===========================================================================
# Cell 2 — Build the 3 log+GVZ+macro design tables (mirror HAR_simpleOLS_3d_with_macro)
# ===========================================================================
for col in ["RV_gold", "GVZ_close", "RV_ES", "RV_crude"]:
    assert (data[col] > 0).all(), f"{col} has non-positive values; log undefined"

x = np.log(rv)

def build_log_design(extra_cols):
    df = pd.DataFrame(index=rv.index)
    df["x_d"] = x
    df["x_w"] = x.rolling(5).mean()
    df["x_m"] = x.rolling(22).mean()
    for name, series in extra_cols.items():
        df[name] = series.reindex(rv.index)
    df["y_log"]   = x.shift(-1)        # log(RV_{t+1})
    df["y_level"] = rv.shift(-1)       # RV_{t+1} in levels, for QLIKE
    return df.dropna()

log_gvz   = np.log(data["GVZ_close"])
log_spx   = np.log(data["RV_ES"])
log_crude = np.log(data["RV_crude"])
macro     = data["macro_event"].shift(-1).astype(float)   # day-(t+1) scheduled-release dummy

d_gvz_macro       = build_log_design({"log_GVZ": log_gvz, "macro": macro})
d_crude_gvz_macro = build_log_design({"log_GVZ": log_gvz, "log_RV_crude": log_crude, "macro": macro})
d_spx_gvz_macro   = build_log_design({"log_GVZ": log_gvz, "log_RV_ES": log_spx, "macro": macro})

# All three designs share the same dropna index -> per-day losses align for the MCS.
assert d_gvz_macro.index.equals(d_crude_gvz_macro.index)
assert d_gvz_macro.index.equals(d_spx_gvz_macro.index)
print("Run 18  log+GVZ+macro        cols:", list(d_gvz_macro.columns),       f"({len(d_gvz_macro)} rows)")
print("Run 20  log+crude+GVZ+macro  cols:", list(d_crude_gvz_macro.columns), f"({len(d_crude_gvz_macro)} rows)")
print("Run 19  log+SPX+GVZ+macro    cols:", list(d_spx_gvz_macro.columns),   f"({len(d_spx_gvz_macro)} rows)")

In [ ]:
# ===========================================================================
# Cell 3 — Shared helpers: recency weights, QLIKE, common-OOS start, HAR loss series
# ===========================================================================
# Common OOS = forecast dates on/after the longest reference window has warmed up. Using
# the same gate as HAR_simpleOLS_3d_with_macro keeps these losses aligned with that
# notebook's MCS surface (sanity check below).
START_DATE = d_gvz_macro.index[max(WINDOWS)]

def _recency_weights(n, delta):
    """Geometric recency weights: newest row weight 1, row `age` older -> delta**age,
    mean-1 normalised. delta=1.0 -> equal weights (unweighted base case)."""
    if delta >= 1.0:
        return np.ones(n)
    ages = np.arange(n)[::-1]
    w = delta ** ages
    return w * (n / w.sum())

def _qlike(actual, forecast, eps=EPS):
    f = np.maximum(forecast, eps)
    r = actual / f
    return r - np.log(r) - 1.0, int((forecast <= eps).sum())

def rolling_log_ols_loss_series(design, feat_cols, window=WINDOW, delta=DELTA, start_date=None):
    """Vectorised weighted-OLS HAR: per-day QLIKE (levels) as a date-indexed Series.
    Copied from HAR_simpleOLS_3d_with_macro.ipynb."""
    if start_date is None:
        start_date = START_DATE
    X = np.column_stack([np.ones(len(design)), design[feat_cols].to_numpy()])
    yl  = design["y_log"].to_numpy()
    lvl = design["y_level"].to_numpy()
    idx = design.index
    N, p = X.shape
    t_all = np.arange(window, N)
    t_oos = t_all[idx[window:] >= start_date]
    starts = t_oos - window
    Xwins = np.lib.stride_tricks.sliding_window_view(X, window, axis=0)[starts].transpose(0, 2, 1)
    ywins = np.lib.stride_tricks.sliding_window_view(yl, window)[starts]
    w  = _recency_weights(window, delta); sw = np.sqrt(w)
    Xs = Xwins * sw[None, :, None]; ys = ywins * sw[None, :]
    A = np.einsum("nwi,nwj->nij", Xs, Xs)
    b = np.einsum("nwi,nw->ni", Xs, ys)
    beta = np.linalg.solve(A, b)
    fitted = np.einsum("nwp,np->nw", Xwins, beta)
    smear = np.einsum("nw,w->n", np.exp(ywins - fitted), w) / w.sum()
    x_pred = np.einsum("np,np->n", X[t_oos], beta)
    fc = np.exp(x_pred) * smear
    q, _ = _qlike(lvl[t_oos], fc)
    return pd.Series(q, index=idx[t_oos], name="qlike")

print(f"Common OOS start: {START_DATE.date()}  "
      f"({int((d_gvz_macro.index >= START_DATE).sum())} forecast days)")

In [ ]:
# ===========================================================================
# Cell 4 — HAR baselines (3 specs) at 6y / delta=0.999
# ===========================================================================
har_specs = [
    ("har_run18", d_gvz_macro,       ["x_d", "x_w", "x_m", "log_GVZ", "macro"]),
    ("har_run19", d_spx_gvz_macro,   ["x_d", "x_w", "x_m", "log_GVZ", "log_RV_ES", "macro"]),
    ("har_run20", d_crude_gvz_macro, ["x_d", "x_w", "x_m", "log_GVZ", "log_RV_crude", "macro"]),
]
har_losses = {name: rolling_log_ols_loss_series(d, f) for name, d, f in har_specs}
for name in har_losses:
    print(f"{name}: mean QLIKE = {har_losses[name].mean():.6f}  (n={len(har_losses[name])})")

In [ ]:
# ===========================================================================
# Cell 5 — XGBoost walk-forward (daily refit), recency weights via sample_weight
# ===========================================================================
# Mirrors the HAR pipeline exactly: same t_oos forecast origins, same recency weights w,
# log target, and the SAME weighted Duan smearing for the log->level back-transform. The
# only change is OLS -> gradient-boosted trees. Conservative fixed params (shallow trees,
# no per-day tuning) to avoid overfitting the ~1512-row window.
XGB_PARAMS = dict(objective="reg:squarederror", tree_method="hist", max_depth=3,
                  learning_rate=0.05, n_estimators=300, subsample=0.8,
                  colsample_bytree=0.8, min_child_weight=5, reg_lambda=1.0,
                  n_jobs=-1, random_state=42)

def rolling_xgb_loss_series(design, feat_cols, window=WINDOW, delta=DELTA,
                            start_date=None, params=None, label=""):
    if start_date is None:
        start_date = START_DATE
    if params is None:
        params = XGB_PARAMS
    Xall = design[feat_cols].to_numpy()
    yl   = design["y_log"].to_numpy()
    lvl  = design["y_level"].to_numpy()
    idx  = design.index
    N = len(design)
    t_all = np.arange(window, N)
    t_oos = t_all[idx[window:] >= start_date]
    w = _recency_weights(window, delta)            # positional weights, oldest..newest
    fc = np.empty(len(t_oos))
    t0 = time.time()
    for i, t in enumerate(t_oos):
        s = t - window
        Xtr, ytr = Xall[s:t], yl[s:t]              # trailing window rows [s, t)
        m = xgb.XGBRegressor(**params)
        m.fit(Xtr, ytr, sample_weight=w)
        fitted = m.predict(Xtr)                    # in-window fit for Duan smearing
        smear = np.sum(w * np.exp(ytr - fitted)) / w.sum()
        x_pred = m.predict(Xall[t:t+1])[0]         # forecast log RV_{t+1}
        fc[i] = np.exp(x_pred) * smear
        if (i + 1) % 250 == 0:
            print(f"  [{label}] {i+1}/{len(t_oos)}  ({(time.time()-t0)/(i+1)*1000:.0f} ms/fit)")
    q, clip = _qlike(lvl[t_oos], fc)
    print(f"[{label}] done: mean QLIKE={q.mean():.6f}  clipped={clip}  "
          f"elapsed={time.time()-t0:.0f}s")
    return pd.Series(q, index=idx[t_oos], name="qlike")

feats_matched = ["x_d", "x_w", "x_m", "log_GVZ", "macro"]
feats_rich    = ["x_d", "x_w", "x_m", "log_GVZ", "log_RV_crude", "log_RV_ES", "macro"]

xgb_matched      = rolling_xgb_loss_series(d_gvz_macro,       feats_matched, delta=DELTA, label="xgb_matched")
xgb_rich         = rolling_xgb_loss_series(d_crude_gvz_macro, feats_rich,    delta=DELTA, label="xgb_rich")
xgb_matched_flat = rolling_xgb_loss_series(d_gvz_macro,       feats_matched, delta=1.0,   label="xgb_matched_d1.0")

In [ ]:
# ===========================================================================
# Cell 6 — Assemble aligned loss matrix, cache, and summary table
# ===========================================================================
all_losses = {
    "xgb_matched": xgb_matched,
    "xgb_rich":    xgb_rich,
    "har_run18":   har_losses["har_run18"],
    "har_run19":   har_losses["har_run19"],
    "har_run20":   har_losses["har_run20"],
}
losses = pd.DataFrame(all_losses)
# All models share identical OOS dates -> no rows should drop.
assert losses.notna().all().all(), "misaligned OOS dates between models"
n_before = len(losses)
losses = losses.dropna()
assert len(losses) == n_before and len(losses) > 0
losses.to_parquet("xgb_vs_har_losses_6y.parquet")
print(f"Aligned loss matrix: {losses.shape[0]} OOS days x {losses.shape[1]} models  "
      f"({losses.index.min().date()} .. {losses.index.max().date()})")

# Recency-effect check: does sample_weight actually change the XGBoost fit?
print(f"\nXGBoost recency effect (matched features):"
      f"\n  delta=1.000 (flat)   mean QLIKE = {xgb_matched_flat.mean():.6f}"
      f"\n  delta=0.999 (recency) mean QLIKE = {xgb_matched.mean():.6f}"
      f"\n  improvement from recency weighting = {xgb_matched_flat.mean() - xgb_matched.mean():+.6f}")

summary = losses.mean().sort_values().rename("mean_qlike").to_frame()
print("\nMean QLIKE by model (lower = better):")
print(summary.to_string())

In [ ]:
# ===========================================================================
# Cell 7 — Model Confidence Set (Hansen, Lunde & Nason 2011)
# ===========================================================================
# p-value > 0.05 => model is in the 5% MCS (statistically indistinguishable from the
# best). Stationary bootstrap handles serial dependence in the daily QLIKE losses.
mcs = MCS(losses, size=0.05, reps=10000, block_size=None,
          method="R", bootstrap="stationary", seed=42)
mcs.compute()

pv = mcs.pvalues["Pvalue"]
mcs_results = (pd.DataFrame({"mean_qlike": losses.mean(), "mcs_pvalue": pv})
               .assign(in_mcs=lambda d: d["mcs_pvalue"] > 0.05)
               .sort_values("mean_qlike"))
mcs_results.to_parquet("mcs_xgb_vs_har_6y.parquet")

pd.set_option("display.float_format", lambda v: f"{v:.6f}")
print("Model Confidence Set (size=0.05, reps=10000, stationary bootstrap):\n")
print(mcs_results.to_string())
print(f"\nBest (lowest mean QLIKE): {mcs_results.index[0]}")
print(f"In 5% MCS: {list(mcs_results.index[mcs_results['in_mcs']])}")
print(f"Excluded:  {list(mcs_results.index[~mcs_results['in_mcs']])}")
mcs_results

In [ ]:
# ===========================================================================
# Cell 8 — When does each model win? Cumulative QLIKE difference vs HAR Run 18
# ===========================================================================
# Cumulative sum of (xgb_matched - har_run18) daily QLIKE. Rising => HAR better over
# that stretch; falling => XGBoost better.
fig, ax = plt.subplots(figsize=(11, 5))
for name, color in [("xgb_matched", "C0"), ("xgb_rich", "C1"),
                    ("har_run19", "C2"), ("har_run20", "C3")]:
    ax.plot(losses.index, (losses[name] - losses["har_run18"]).cumsum(),
            label=f"{name} - har_run18", color=color, lw=1.2)
ax.axhline(0, color="k", lw=0.8)
ax.set_title("Cumulative QLIKE difference vs HAR Run 18 (6y, δ=0.999)\n"
             "below 0 = model beats HAR Run 18 cumulatively")
ax.set_ylabel("cumulative ΔQLIKE")
ax.legend()
fig.tight_layout()
fig.savefig("xgb_vs_har_cum_qlike_6y.png", dpi=150)
plt.show()